# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Moho Goswami

**ID**: mg2297

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\mohog\hw5-mohog`
   Installed Measures ─────────── v0.3.3
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed GR_jll ───────────── v0.73.18+0
   Installed PlotUtils ────────── v1.4.4
   Installed OpenSSL ──────────── v1.6.0
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed MutableArithmetics ─ v1.6.7
   Installed Pango_jll ────────── v1.57.0+0
   Installed FFMPEG ───────────── v0.4.5
   Installed StaticArraysCore ─── v1.4.4
   Installed JSON ─────────────── v1.3.0
   Installed DataStructures ───── v0.19.3
   Installed GraphRecipes ─────── v0.5.15
   Installed METIS_jll ────────── v5.1.3+0
   Installed StatsBase ────────── v0.34.8
   Installed FFMPEG_jll ───────── v8.0.0+0
   Installed StableRNGs ───────── v1.0.4
   Installed HiGHS ────────────── v1.20.1
   Installed ForwardDiff ──────── v1.3.0
   Installed StructUtils ──────── v2.6.0
   Installed JuMP ─────────────── v1.29.3
   Installed GR ───────────────── v0.73.18
Precompiling project...

In [3]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

#### City 1
* 100 Mg/day solid waste
* Food waste: 15% * 100 = 15 Mg/day
* Paper & Cardboard: 40% * 100 = 40 Mg/day
* Plastics: 5% * 100 = 5 Mg/day
* Textiles: 3% * 100 = 3 Mg/day
* Rubber/leather: 2% * 100 = 2 Mg/day
* Wood: 5% * 100 = 5 Mg/day
* Yard wastes: 18% * 100 = 18 Mg/day
* Glass: 4% * 100 = 4 Mg/day
* Ferrous: 2% * 100 = 2 Mg/day
* Aluminum: 2% * 100 = 2 Mg/day
* Other metal: 1% * 100 = 1 Mg/day
* Misc: 3% * 100 = 3 Mg/day

##### Overall recycling fraction
$$
0\% \cdot 15 +
55\% \cdot 40 +
15\% \cdot 5 +
10\% \cdot 3 +
0\% \cdot 2 +
30\% \cdot 5 +
40\% \cdot 18 +
60\% \cdot 4 +
75\% \cdot 2 +
80\% \cdot 2 +
50\% \cdot 1 +
0\% \cdot 3
$$

$$
= 37.50 Mg/day
$$

##### Overall ash fraction
$$
8\% \cdot 15 +
7\% \cdot 40 +
5\% \cdot 5 +
10\% \cdot 3 +
15\% \cdot 2 +
2\% \cdot 5 +
2\% \cdot 18 +
100\% \cdot 4 +
100\% \cdot 2 +
100\% \cdot 2 +
100\% \cdot 1 +
70\% \cdot 3
$$

$$
= 16.41 Mg/day
$$

#### City 2
* 90 Mg/day solid waste
* Food waste: 15% * 90 = 13.5 Mg/day
* Paper & Cardboard: 40% * 90 = 36 Mg/day
* Plastics: 5% * 90 = 4.5 Mg/day
* Textiles: 3% * 90 = 2.7 Mg/day
* Rubber/leather: 2% * 90 = 1.8 Mg/day
* Wood: 5% * 90 = 4.5 Mg/day
* Yard wastes: 18% * 90 = 16.2 Mg/day
* Glass: 4% * 90 = 3.6 Mg/day
* Ferrous: 2% * 90 = 1.8 Mg/day
* Aluminum: 2% * 90 = 1.8 Mg/day
* Other metal: 1% * 90 = 0.9 Mg/day
* Misc: 3% * 90 = 2.7 Mg/day

##### Overall recycling fraction
$$
0\% \cdot 13.5 +
55\% \cdot 36 +
15\% \cdot 4.5 +
10\% \cdot 2.7 +
0\% \cdot 1.8 +
30\% \cdot 4.5 +
40\% \cdot 16.2 +
60\% \cdot 3.6 +
75\% \cdot 1.8 +
80\% \cdot 1.8 +
50\% \cdot 0.9 +
0\% \cdot 2.7
$$

$$
= 33.975 Mg/day
$$

##### Overall ash fraction
$$
8\% \cdot 13.5 +
7\% \cdot 36 +
5\% \cdot 4.5 +
10\% \cdot 2.7 +
15\% \cdot 1.8 +
2\% \cdot 4.5 +
2\% \cdot 16.2 +
100\% \cdot 3.6 +
100\% \cdot 1.8 +
100\% \cdot 1.8 +
100\% \cdot 0.9 +
70\% \cdot 2.7
$$

$$
= 14.769 Mg/day
$$

#### City 3
* 120 Mg/day solid waste
* Food waste: 15% * 120 = 18 Mg/day
* Paper & Cardboard: 40% * 120 = 48 Mg/day
* Plastics: 5% * 120 = 6 Mg/day
* Textiles: 3% * 120 = 3.6 Mg/day
* Rubber/leather: 2% * 120 = 2.4 Mg/day
* Wood: 5% * 120 = 6 Mg/day
* Yard wastes: 18% * 120 = 21.6 Mg/day
* Glass: 4% * 120 = 4.8 Mg/day
* Ferrous: 2% * 120 = 2.4 Mg/day
* Aluminum: 2% * 120 = 2.4 Mg/day
* Other metal: 1% * 120 = 1.2 Mg/day
* Misc: 3% * 120 = 3.6 Mg/day

##### Overall recycling fraction
$$
0\% \cdot 18 +
55\% \cdot 48 +
15\% \cdot 6 +
10\% \cdot 3.6 +
0\% \cdot 2.4 +
30\% \cdot 6 +
40\% \cdot 21.6 +
60\% \cdot 4.8 +
75\% \cdot 2.4 +
80\% \cdot 2.4 +
50\% \cdot 1.2 +
0\% \cdot 3.6
$$

$$
= 45.3 Mg/day
$$
##### Overall ash fraction
$$
8\% \cdot 18 +
7\% \cdot 48 +
5\% \cdot 6 +
10\% \cdot 3.6 +
15\% \cdot 2.4 +
2\% \cdot 6 +
2\% \cdot 21.6 +
100\% \cdot 4.8 +
100\% \cdot 2.4 +
100\% \cdot 2.4 +
100\% \cdot 1.2 +
70\% \cdot 3.6
$$

$$
= 19.692 Mg/day
$$

$$
\begin{array}{l r r}
\text{City} & \text{Recycling (Mg/day)} & \text{Ash (Mg/day)} \\
\hline
\text{City 1} & 37.50 & 16.41 \\
\text{City 2} & 33.975 & 14.769 \\
\text{City 3} & 45.30 & 19.692 \\
\end{array}
$$

#### From the table above, the actual percentages that we'll need for the next questions are a recycling percentage of 37.50% and a combustion ash percentage of 16.41%

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

In waste management optimization problems such as this one, there are 3 key categories of decision variables: waste transported, residual waste transported, and operational status of each disposal type.

$$
Cities: i ∈ {1, 2, 3}
$$

$$
Disposals: i ∈ {L, M, W}
$$

\begin{array}{l l l}
\text{Variable} & \text{Definition} & \text{Description} \\
\hline
W_{1,L} & \text{Waste from City 1 to LF} & \text{City 1 waste transported to site L (Mg/day)} \\
W_{1,M} & \text{Waste from City 1 to MRF} & \text{City 1 waste transported to site M (Mg/day)} \\
W_{1,W} & \text{Waste from City 1 to WTE} & \text{City 1 waste transported to site W (Mg/day)} \\
W_{2,L} & \text{Waste from City 2 to LF} & \text{City 2 waste transported to site L (Mg/day)} \\
W_{2,M} & \text{Waste from City 2 to MRF} & \text{City 2 waste transported to site M (Mg/day)} \\
W_{2,W} & \text{Waste from City 2 to WTE} & \text{City 2 waste transported to site W (Mg/day)} \\
W_{3,L} & \text{Waste from City 3 to LF} & \text{City 3 waste transported to site L (Mg/day)} \\
W_{3,M} & \text{Waste from City 3 to MRF} & \text{City 3 waste transported to site M (Mg/day)} \\
W_{3,W} & \text{Waste from City 3 to WTE} & \text{City 3 waste transported to site W (Mg/day)} \\
\hline
R_{L,M} & \text{Residual waste from L to M} & \text{Residuals from LF to MRF (Mg/day)} \\
R_{L,W} & \text{Residual waste from L to W} & \text{Residuals from LF to WTE (Mg/day)} \\
R_{M,L} & \text{Residual waste from M to L} & \text{Residuals from MRF to LF (Mg/day)} \\
R_{M,W} & \text{Residual waste from M to W} & \text{Residuals from MRF to WTE (Mg/day)} \\
R_{W,L} & \text{Residual waste from W to L} & \text{Residuals from WTE to LF (Mg/day)} \\
R_{W,M} & \text{Residual waste from W to M} & \text{Residuals from WTE to MRF (Mg/day)} \\
\hline
Y_{L} & \text{Operational status of LF} & \text{1 if LF is active, 0 otherwise} \\
Y_{M} & \text{Operational status of MRF} & \text{1 if MRF is active, 0 otherwise} \\
Y_{W} & \text{Operational status of WTE} & \text{1 if WTE is active, 0 otherwise} \\
\end{array}


#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

##### The following generic objective function seeks to minimize the total cost; the first term covers the transportation cost between sources and facilities while the second covers disposal costs accounting for facilities operating.

$$
\min_{W_{ij},\, Y_j} 
\Bigg\{
\sum_i \sum_j a_{ij} \, l_{ij} \, W_{ij} 
\;+\; 
\sum_j \Big( c_j Y_j + b_j \sum_i W_{ij} \Big)
\Bigg\}
$$

Using the problem data, the formula can be rewritten using specific coefficients and variables.

##### LF Costs
$ 2000 + 50(W_{1, L} + W_{2, L} + W_{3, L} + R_{M, L} + R_{W, L}) $

##### MSW Costs
$ 1500 + 7(W_{1, M} + W_{2, M} + W_{3, M} + R_{L, M} + R_{W, M}) + (0.3750)(40)(W_{1, M}) + (0.33975)(40)(W_{2, M}) + (0.4530)(40)(W_{3, M})$

##### WTE Costs
$ 2500 + 60(W_{1, W} + W_{2, W} + W_{3, W} + R_{L, W} + R_{M, W}) $

##### Transportation Costs
$ 1.5(5W_{1, L} + 30W_{1, M} + 15W_{1, W} +  15W_{2, L} + 25W_{2, M} + 10W_{2, W} + 13W_{3, L} + 45W_{3, M} + 20W_{3, W} + 32R_{L, M} + 18W_{L, W} + 15W_{M, W})$

The objective function can be derived by summing all four categories of the cost terms.

##### Objective function
$$
\min_{W, R, Y} 
6000 + 57.5(W_{1,L}) + 72.5(W_{2,L}) + 69.5(W_{3,L}) + 50(R_{M,L}) 
\\
+ 50(R_{W,L}) + 67(W_{1,M}) + 58.09(W_{2,M}) + 92.62(W_{3,M}) + 55(R_{L,M}) 
\\
+ 7(R_{W,M}) + 22.5(W_{M,W}) + 82.5(W_{1,W}) + 75(W_{2,W}) 
\\
+ 90(W_{3,W}) + 60(R_{L,W}) + 60(R_{M,W}) + 27(W_{L,W})
$$



#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

#### City mass balance constraints
* $ W_{1, L} + W_{1, M} + W_{1, W} = 100 $
* $ W_{2, L} + W_{2, M} + W_{2, W} = 90 $
* $ W_{3, L} + W_{3, M} + W_{3, W} = 120 $

#### Residual mass balance constraints
* $ R_{W, L} = 0.1641(W_{1, L} + W_{2, L} + R_{M, W}) $
* $ R_{M, W} + R_{M, L} = 0.5391*(W_{1, L} + W_{2, L}) $

#### Disposal limit constraints
* $ W_{1, L} + W_{2, L} + W_{3, L} \le 200 $
* $ W_{1, M} + W_{2, M} + W_{3, M} \le 350 $
* $ W_{1, W} + W_{2, W} + W_{3, W} \le 210 $

#### Non-negativity
* $ W_{i,j}, R_{i, j} \ge 0 $

#### Operating status
* $ Y_{j} ∈ [0, 1] $

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [7]:
using JuMP, HiGHS

In [12]:
cities = 1:3
Q = Dict(1 => 100.0, 2 => 90.0, 3 => 120.0)
d = Dict(
    :L => Dict(1=>5.0,  2=>15.0, 3=>13.0),
    :M => Dict(1=>30.0, 2=>25.0, 3=>45.0),
    :W => Dict(1=>15.0, 2=>10.0, 3=>20.0)
)

dist_L_M = 32.0
dist_L_W = 18.0
dist_M_W = 15.0

c_trans = 1.5
c_L = 50.0
c_M = 7.0
c_recycle = 40.0
recycle_frac = 0.3750
ash_frac = 0.1641
combo_frac = 0.5391

c_W = 60.0

F_L = 2000.0
F_M = 1500.0
F_W = 2500.0

cap_L = 200.0
cap_M = 350.0
cap_W = 210.0

model = Model(HiGHS.Optimizer)

@variable(model, W_L[i in cities] >= 0)
@variable(model, W_M[i in cities] >= 0)
@variable(model, W_W[i in cities] >= 0)

@variable(model, R_M_L >= 0)   # R_{M,L}
@variable(model, R_W_L >= 0)   # R_{W,L}
@variable(model, R_M_W >= 0)   # R_{M,W}
@variable(model, R_L_M >= 0)   # R_{L,M}
@variable(model, R_L_W >= 0)   # R_{L,W}

@variable(model, yL, Bin)
@variable(model, yM, Bin)
@variable(model, yW, Bin)

# transport from city to facilities
transport_city = sum(c_trans * ( d[:L][i]*W_L[i] + d[:M][i]*W_M[i] + d[:W][i]*W_W[i] ) for i in cities)
transport_interfac = c_trans * ( dist_L_M * R_L_M + dist_L_W * R_L_W + dist_M_W * R_M_W )

tipping_L = c_L * ( sum(W_L[i] for i in cities) + R_M_L + R_W_L )
tipping_M = c_M * ( sum(W_M[i] for i in cities) + R_L_M + 0.0 )
tipping_W = c_W * ( sum(W_W[i] for i in cities) + R_L_W + R_M_W )

recycle_proc = sum( recycle_frac * c_recycle * W_M[i] for i in cities )
fixed_costs = F_L * yL + F_M * yM + F_W * yW

@objective(model, Min,
    transport_city + transport_interfac +
    tipping_L + tipping_M + tipping_W +
    recycle_proc + fixed_costs
)

@constraint(model, [i in cities], W_L[i] + W_M[i] + W_W[i] == Q[i])

@constraint(model, R_W_L == ash_frac * ( W_L[1] + W_L[2] + R_M_W ))
@constraint(model, R_M_W + R_M_L == combo_frac * ( W_L[1] + W_L[2] ))

@constraint(model, sum(W_L[i] for i in cities) + R_M_L + R_W_L <= cap_L * yL)
@constraint(model, sum(W_M[i] for i in cities) + R_L_M <= cap_M * yM)
@constraint(model, sum(W_W[i] for i in cities) + R_L_W + R_M_W <= cap_W * yW)

optimize!(model)


Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 8 rows; 17 cols; 34 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [2e-01, 4e+02]
  Cost    [5e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
8 rows, 15 cols, 32 nonzeros  0s
7 rows, 14 cols, 26 nonzeros  0s
Presolve reductions: rows 7(-1); columns 14(-3); nonzeros 26(-8) 

Solving MIP model with:
   7 rows
   14 cols (3 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   26 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective 

In [13]:
println("\nTermination status: ", termination_status(model))
println("Optimal objective value (cost per day): ",
        round(objective_value(model); digits = 3))

println("\nCity → Facility flows (Mg/day)")
for i in cities
    println("City ", i, " → Landfill (W_", i, ",L): ", round(value(W_L[i]); digits=3))
    println("City ", i, " → MRF      (W_", i, ",M): ", round(value(W_M[i]); digits=3))
    println("City ", i, " → WTE      (W_", i, ",W): ", round(value(W_W[i]); digits=3))
    println("")
end

println("Facility openings and inter-facility flows")
println("yL (Landfill open) = ", value(yL))
println("yM (MRF open)     = ", value(yM))
println("yW (WTE open)     = ", value(yW))
println("R_M_L = ", round(value(R_M_L); digits=4))
println("R_W_L = ", round(value(R_W_L); digits=4))
println("R_M_W = ", round(value(R_M_W); digits=4))
println("R_L_M = ", round(value(R_L_M); digits=4))
println("R_L_W = ", round(value(R_L_W); digits=4))


Termination status: OPTIMAL
Optimal objective value (cost per day): 23895.0

City → Facility flows (Mg/day)
City 1 → Landfill (W_1,L): 0.0
City 1 → MRF      (W_1,M): 100.0
City 1 → WTE      (W_1,W): 0.0

City 2 → Landfill (W_2,L): 0.0
City 2 → MRF      (W_2,M): 90.0
City 2 → WTE      (W_2,W): 0.0

City 3 → Landfill (W_3,L): 120.0
City 3 → MRF      (W_3,M): 0.0
City 3 → WTE      (W_3,W): 0.0

Facility openings and inter-facility flows
yL (Landfill open) = 1.0
yM (MRF open)     = 1.0
yW (WTE open)     = -0.0
R_M_L = 0.0
R_W_L = 0.0
R_M_W = 0.0
R_L_M = 0.0
R_L_W = 0.0


In [14]:
# cost breakdown (component values)
println("\n--- Cost breakdown (components) ---")
println("Transport (city->facilities): ", round(value(transport_city); digits=3))
println("Transport (inter-facility):  ", round(value(transport_interfac); digits=3))
println("Tipping L: ", round(value(tipping_L); digits=3))
println("Tipping M: ", round(value(tipping_M); digits=3))
println("Tipping W: ", round(value(tipping_W); digits=3))
println("Recycling processing: ", round(value(recycle_proc); digits=3))
println("Fixed costs: ", round(value(fixed_costs); digits=3))
println("Sum check (should equal objective): ", round(
    value(transport_city) + value(transport_interfac) +
    value(tipping_L) + value(tipping_M) + value(tipping_W) +
    value(recycle_proc) + value(fixed_costs); digits=3))


--- Cost breakdown (components) ---
Transport (city->facilities): 10215.0
Transport (inter-facility):  0.0
Tipping L: 6000.0
Tipping M: 1330.0
Tipping W: 0.0
Recycling processing: 2850.0
Fixed costs: 3500.0
Sum check (should equal objective): 23895.0


#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

### Data from CSV file
$$
\begin{array}{l l r r r r}
\text{Plant} & \text{Resource} & P_{\min} & P_{\max} & \text{VarCost} & \text{Ramp} \\
\hline
\text{Biomass} & \text{Biomass} & 0 & 100 & 5 & 100 \\
\text{Hydroelectric} & \text{Hydroelectric} & 0 & 500 & 0 & 500 \\
\text{Geothermal} & \text{Geothermal} & 0 & 0 & 400 & 400 \\
\text{NG CCGT} & \text{NG CCGT} & 220 & 500 & 23 & 100 \\
\text{NG CT} & \text{NG CT} & 100 & 250 & 38 & 200 \\
\text{Wind} & \text{Wind} & 0 & 300 & 0 & 300 \\
\text{Solar} & \text{Solar} & 0 & 500 & 0 & 500 \\
\end{array}
$$


#### Decision variables
Demand
$$
d_{i}, where \\
d_{1} = 1100\ \text{MW} \\
d_{2,1} = 1200\ \text{MW} \\
d_{2,2} = 1200\ \text{MW} \\
d_{2,3} = 1500\ \text{MW} \\
d_{2,4} = 1500\ \text{MW} \\
$$

Decision variables: Amount of each source in MW, total sum must meet demand
$$
\sum_{g \in G} y_{1,g} = d_{1} \\
\sum_{g \in G} y_{2_g, 1} = d_{2,1} \\
\sum_{g \in G} y_{2_g, 2} = d_{2,2} \\
\sum_{g \in G} y_{2_g, 3} = d_{2,3} \\
\sum_{g \in G} y_{2_g, 4} = d_{2,4} \\
$$

where g represents generator type
* $y_{2_g, 1}$ = scenario with period 2, 1200 MW demand, 70% likely capacity factors
* $y_{2_g, 2}$ = scenario with period 2, 1200 MW demand, 30% likely capacity factors
* $y_{2_g, 3}$ = scenario with period 2, 1500 MW demand, 70% likely capacity factors
* $y_{2_g, 4}$ = scenario with period 2, 1500 MW demand, 30% likely capacity factors

#### Objective Function: Minimize Cost

$$
min \sum_{i} VarCost_gy_{1,g} + 0.525\sum_{i} VarCost_gy_{2,g,1} + 0.225\sum_{i} VarCost_gy_{2,g,2} + 0.175\sum_{i} VarCost_gy_{2,g,3} + 0.075\sum_{i} VarCost_gy_{2,g,4}
$$

#### Output Constraints, Period 1
$$
y_{1, biomass} \le 100\ \text{MW} \\
y_{1, hydro} \le 500\ \text{MW} \\
y_{1, geothermal} \le 0\ \text{MW} \\
y_{1, ngccgt} \le 500\ \text{MW} \\
y_{1, ngct} \le 250\ \text{MW} \\
y_{1, wind} \le 135 \ \text{MW} \\
y_{1, solar} \le 450 \ \text{MW} \\
$$

$$
y_{1, biomass} \ge 0\ \text{MW} \\
y_{1, hydro} \ge 0\ \text{MW} \\
y_{1, geothermal} \ge 0\ \text{MW} \\
y_{1, ngccgt} \ge 220\ \text{MW} \\
y_{1, ngct} \ge 100\ \text{MW} \\
y_{1, wind} \ge 0 \ \text{MW} \\
y_{1, solar} \ge 0 \ \text{MW} \\
$$

#### Output Constraints, Period 2-Specific
$$
y_{2, biomass} \le 100\ \text{MW} \\
y_{2, hydro} \le 500\ \text{MW} \\
y_{2, geothermal} \le 0\ \text{MW} \\
y_{2, ngccgt} \le 500\ \text{MW} \\
y_{2, ngct} \le 250\ \text{MW} \\
y_{2, wind, 1} \le 120\ \text{MW} \\
y_{2, wind, 2} \le 150\ \text{MW} \\
y_{2, solar, 1} \le 475\ \text{MW} \\
y_{2, solar, 2} \le 375\ \text{MW} \\
$$

$$
y_{2, biomass} \ge 0\ \text{MW} \\
y_{2, hydro} \ge 0\ \text{MW} \\
y_{2, geothermal} \ge 0\ \text{MW} \\
y_{2, ngccgt} \ge 220\ \text{MW} \\
y_{2, ngct} \ge 100\ \text{MW} \\
y_{2, wind} \ge 0\ \text{MW} \\
y_{2, solar} \ge 0\ \text{MW} \\
$$

#### Ramping and Non-Negativity Constraints
The non-negativity constraints, setting the minimum MW amount from each source to be at least, if not more than, 0 have already been accounted for via the minimum power output constraints above.

$$
-100 \le y_{2, biomass} - y_{1, biomass} \le 100 \\
-500 \le y_{2, hydro} - y_{1, hydro} \le 500 \\
-400 \le y_{2, geothermal} - y_{1, geothermal} \le 400 \\
-100 \le y_{2, ngccgt} - y_{1, ngccgt} \le 100 \\
-200 \le y_{2, ngct} - y_{1, ngct} \le 200 \\
-300 \le y_{2, wind} - y_{1, wind} \le 300 \\
-500 \le y_{2, solar} - y_{1, solar} \le 500 \\
$$

## References

List any external references consulted, including classmates.